# rete-graph, live in your browser

**Everything in this notebook runs in your browser tab.** The Python you are
using is [Pyodide](https://pyodide.org) (CPython compiled to WebAssembly),
the notebook UI is [JupyterLite](https://jupyterlite.readthedocs.io) — and
the package under test is [`rete-graph`](https://pypi.org/project/rete-graph/),
whose PyEmscripten wheel installs straight from PyPI.

A `.rete` file is a single, immutable, **range-queryable RDF graph**: it sits
on plain object storage, and the client fetches only the byte ranges a SPARQL
query touches. No database server anywhere — not even outside this tab.

Run the cells top to bottom (`Shift+Enter`). The first one downloads the
wheel (~2 MB) and pandas, so give it a moment.

Project: <https://github.com/caviri/rete> · docs:
[Python API](https://caviri.github.io/rete/python.html)

In [ ]:
%pip install rete-graph pandas

import sys
import rete_graph as rete
print(f"rete-graph {rete.__version__} running on platform: {sys.platform!r}")

## Open a remote graph — lazily

The [BOE legal graph](https://caviri.github.io/rete/playground.html) (Spanish
state bulletin): **447,128 triples, 6.9 MB**, hosted on Cloudflare R2. Opening
it fetches only the header, dictionary directory, and index directories —
the kernel runs in a web worker, so the client's synchronous `XMLHttpRequest`
range reads are allowed.

In [ ]:
g = rete.open("https://data.graphplaza.com/boe/boe.rete")
s = g.stats()
print(f"{g.quads:,} triples available; opened after fetching just "
      f"{s['bytes']:,} of {s['fileLength']:,} bytes in {s['requests']} range requests")

## SPARQL → pandas

`query_df()` runs a SPARQL SELECT and hands back a **pandas DataFrame**, so
results render as proper tables right here. This query lists laws that cite
the most other laws:

In [ ]:
g.query_df("""
    SELECT ?law (COUNT(?cited) AS ?cites) WHERE {
        ?law <http://data.europa.eu/eli/ontology#cites> ?cited
    }
    GROUP BY ?law ORDER BY DESC(?cites) LIMIT 10
""")

## The dataset documents itself

A `.rete` file can embed its own **Dataset Card** with a library of starter
queries. `examples()` reads them with one small ranged request — the table
below comes from *inside the file*, and every `sparql` entry is runnable:

In [ ]:
import pandas as pd

examples = g.examples()
pd.DataFrame(examples)[["tier", "title", "question"]]

In [ ]:
# ... so let's run the first one, exactly as shipped in the file:
print(examples[0]["title"], "—", examples[0]["question"])
g.query_df(examples[0]["sparql"])

## What's in the graph?

The schema profile — which classes exist and how often — as another table:

In [ ]:
pd.DataFrame(g.schema()["classes"], columns=["class", "instances"])

## Build a graph of your own — in the browser

The builder works here too: RDF in, an immutable `.rete` image out, with a
Dataset Card embedded. (For big datasets, use the
[`rete build` CLI](https://caviri.github.io/rete/cli.html).)

In [ ]:
mine = (
    rete.Builder()
    .add('<urn:tab> <urn:runs> <urn:sparql> .\n<urn:tab> <urn:needs> <urn:no_server> .')
    .card(title="Built inside a browser tab", license="CC0-1.0")
    .example("SELECT ?s ?p ?o WHERE { ?s ?p ?o }", title="Everything")
    .graph()
)
print(mine.card()["title"], "·", mine.quads, "triples")
mine.query_df(mine.examples()[0]["sparql"])

---

**Where next:** the [playground](https://caviri.github.io/rete/playground.html)
(40+ datasets, no code) · [Python API](https://caviri.github.io/rete/python.html) ·
[build tutorial](https://caviri.github.io/rete/python-build-tutorial.html) ·
[JavaScript client](https://caviri.github.io/rete/javascript.html)

Source & issues: **<https://github.com/caviri/rete>**

© 2026 Carlos Vivar Ríos — released under the
[Apache License 2.0](https://github.com/caviri/rete/blob/main/LICENSE).
Datasets keep their own licenses (BOE: © Agencia Estatal Boletín Oficial del
Estado, [reuse conditions](https://www.boe.es/datosabiertos/)).